# Objects in Focus — tutorial

**What was this person looking at?**

An eye-tracker tells you *where* someone looked: an x and a y. This notebook
takes you from that to *what* they looked at — the sofa, the sink, the third
tree from the left — in scenes where everything is segmented by hand.

Nothing to install on your computer. Run the cells in order with the ▶ button
(or Shift+Enter). The whole thing takes about five minutes.

**What you will do**

1. Install the package and grab a few scenes
2. Look at a scene and its objects
3. Load the study's real fixations and map them onto those objects
4. Build a table with one row per object
5. Fit the attention model from the paper
6. Point it at your own data

Project page: https://ehhall.github.io/objects-in-focus/ ·
Code: https://github.com/ehhall/objects-in-focus

## 1. Setup

This installs the `oif` package. It takes about thirty seconds. Ignore any
message about restarting the runtime — you do not need to.

In [ ]:
!pip install -q "git+https://github.com/ehhall/objects-in-focus.git"

import oif
print("oif version", oif.__version__)

### Get some data

The full dataset is around 500 MB, which is a slow download for a tutorial. So
this cell fetches **six scenes** — the images, the hand-drawn annotations, the
depth maps, **and the real fixations** recorded while people memorized each
scene — and builds the label maps locally. That is the real pipeline, just on
six scenes instead of a hundred.

(Want everything? There is a cell at the bottom of the notebook.)

In [ ]:
import os, urllib.request
from pathlib import Path

SCENES = ["target_livingroom_IDS01", "target_bakery", "target_kitchen_IDS01",
          "target_garden", "target_subway", "target_beach"]

RAW = "https://raw.githubusercontent.com/ehhall/objects-in-focus/main"
root = Path("objects-in-focus")
for folder in ("images", "annotations", "depth", "raw"):
    (root / folder).mkdir(parents=True, exist_ok=True)

def fetch(url, dest):
    if dest.exists():
        return
    urllib.request.urlretrieve(url, dest)

for scene in SCENES:
    fetch(f"{RAW}/images/{scene}.png",           root / "images" / f"{scene}.png")
    fetch(f"{RAW}/annotations/{scene}.xml",      root / "annotations" / f"{scene}.xml")
    fetch(f"{RAW}/depth/{scene}_disp.npy",       root / "depth" / f"{scene}_disp.npy")
    fetch(f"{RAW}/raw/{scene}_memorize.npy",     root / "raw" / f"{scene}_memorize.npy")
    print("got", scene)

print("\ndone -", len(list((root / 'images').glob('*.png'))), "scenes")

## 2. Open a scene

`OiF` is the dataset. Ask it for a scene by name and you get everything that
lines up with that picture.

In [ ]:
from oif import OiF

data = OiF(root)
print(data.scene_names)

scene = data["target_livingroom_IDS01"]
scene

In [ ]:
print("image     ", scene.image.shape, scene.image.dtype)
print("label map ", scene.label_map.shape, "-", len(scene.object_ids), "objects")
print("depth     ", scene.depth.shape, f"{scene.depth.min():.2f} (near) to {scene.depth.max():.2f} (far)")
print()
print("the first few objects:", list(scene.labels.items())[:6])

### What is a label map?

It is an image where each pixel holds a number: the id of the object visible
at that pixel. Zero means no annotated object. So `label_map[y, x]` answers
"what is at this point?", which is exactly the question a fixation asks.

We did not download the label maps — the package built them from the
annotations, painting objects **back to front** using the depth map, so that
when two objects overlap the nearer one wins. That is the object the viewer
actually saw.

In [ ]:
import matplotlib.pyplot as plt
from oif.viz import show_objects, show_scene

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
show_scene(scene.image, ax=axes[0], title="the scene")
show_objects(scene.image, scene.label_map, ax=axes[1],
             title=f"{len(scene.object_ids)} segmented objects")
plt.show()

In [ ]:
# with names written on, which is slower to read but more convincing
fig, ax = plt.subplots(figsize=(12, 9))
show_objects(scene.image, scene.label_map, ax=ax, labels=scene.labels, thickness=2)
plt.show()

## 3. Fixations

These are **real human eye movements**. While the scenes were being
photographed into a memory experiment, viewers studied each one for twelve
seconds, and every place their eyes stopped was recorded.

The release ships them as one file per scene in `raw/` —
`target_bakery_memorize.npy` — a 768 × 1024 array with a 1 at every pixel
where a fixation landed, pooled over all the viewers. Simple, but worth
seeing before we use it:

In [ ]:
import numpy as np

scene = data["target_livingroom_IDS01"]
fixation_map = scene.fixation_map()          # reads raw/<scene>_memorize.npy

print("shape:", fixation_map.shape, "- one cell per image pixel")
print("fixations in this scene:", int(fixation_map.sum()))

fig, ax = plt.subplots(figsize=(10, 7.5))
ax.imshow(scene.image)
ys, xs = np.nonzero(fixation_map)
ax.scatter(xs, ys, s=18, c="#2d7dd2", edgecolors="white", linewidths=0.5)
ax.set_title(f"every recorded fixation on {scene.name}")
ax.axis("off")
plt.show()

### From maps to a table

Arrays are awkward to analyse, so `data.fixations()` reads every file in
`raw/` — these binary maps, or your own CSV exports, or a mix — and returns
one tidy table: `subject, image, fix_index, x, y, duration, task`.

Pooled maps carry no viewer identity, no ordering and no durations, so those
columns come back empty — and every step downstream treats empty as "no
constraint", so nothing breaks. `filter_fixations` applies the standard
cleaning (duration window, first fixation dropped, off-image dropped); with
pooled maps it only has the off-image check left to do, but the same line
works unchanged when your data has the full detail.

In [ ]:
from oif import filter_fixations

fixations = data.fixations()          # every file in raw/
print(len(fixations), "fixations across", fixations["image"].nunique(), "scenes,",
      "task:", fixations["task"].unique().tolist())

fixations = filter_fixations(fixations, shape=scene.shape)
print(len(fixations), "after the standard cleaning steps")
fixations.head()

## 4. Map fixations onto objects

The one line this whole package exists for.

In [ ]:
looked_at = scene.map_fixations(fixations, method="disc", radius=25)
looked_at[["image", "x", "y", "task", "mask_id", "label"]].head(10)

Every fixation now carries the object it landed on.

**`method` is a choice you should make deliberately:**

| method | what it does | when |
|---|---|---|
| `"point"` | the object under the fixation pixel | clean data, large objects |
| `"disc"` | the object covering most of a disc of `radius` px | real data — absorbs calibration drift |
| `"nearest"` | as `point`, but background fixations snap to the nearest object within `radius` | when everything must be attributed |

25 pixels is roughly one degree of visual angle at a normal desktop viewing
distance. Fixations that land on nothing come back as `mask_id = 0`, label
`"background"` — they are never quietly dropped, so counts always add up.

In [ ]:
for method, radius in [("point", 0), ("disc", 25), ("nearest", 25)]:
    mapped = scene.map_fixations(fixations, method=method, radius=radius)
    on_background = (mapped["mask_id"] == 0).mean()
    print(f"{method:8s} radius={radius:<3d} -> {on_background:.1%} of fixations on background")

### See where people looked

In [ ]:
from oif.viz import show_fixations, show_density

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
show_fixations(scene.image, looked_at, ax=axes[0], size=18, title="fixations, colored dots")
show_density(scene.density(fixations), ax=axes[1], title="fixation density")
plt.show()

## 5. One row per object

Fixation-level data answers "what was looked at". Object-level data answers
"what got attention" — and that is what you model.

In [ ]:
table = scene.object_table(fixations, method="disc", radius=25)
table.sort_values("n_fixations", ascending=False).head(8)

The columns:

- **`size`** — visible pixels. Occluded objects count only what shows.
- **`ecc`** — distance in pixels from the object's centre to the centre of the picture.
- **`depth`** — how far away it is, 0 near to 1 far.
- **`salience`** — NaN here; fill it from a salience model if you have one.
- **`n_fixations`** — how many fixations the object collected.
- **`total_duration`, `n_subjects`** — zero here, because pooled maps carry
  no durations or viewer identities; with your own CSV exports these fill in.

Objects nobody looked at stay in the table with zeros. That matters: a model
of how attention spreads across objects has to explain the ones that got none.

In [ ]:
from oif.viz import show_object_values

fig, axes = plt.subplots(1, 3, figsize=(19, 5))
for ax, column, cmap in [(axes[0], "n_fixations", "magma"),
                         (axes[1], "size", "viridis"),
                         (axes[2], "depth", "cividis")]:
    show_object_values(scene.label_map, dict(zip(table["mask_id"], table[column])),
                       ax=ax, cmap=cmap, title=column, colorbar=False)
plt.show()

### The whole dataset at once

In [ ]:
all_objects = data.object_tables(fixations, method="disc", radius=25)
print(len(all_objects), "objects across", all_objects["image"].nunique(), "scenes")
all_objects.groupby("image")["n_fixations"].agg(["count", "sum"]).rename(
    columns={"count": "objects", "sum": "fixations"})

## 6. The attention model

The paper's claim is that you can predict how much attention an object gets
from four things about it: how big it is, how far from the centre, how far
away, and how salient. In model terms:

```
log1p(fixations) ~ log1p(size) + log1p(depth) + z(eccentricity) + z(salience)
```

`add_model_terms` does those transformations for you.

In [ ]:
from oif import add_model_terms, ObjectAttentionModel

table = add_model_terms(all_objects)
model = ObjectAttentionModel().fit(table)

print(model.coefficients.round(3))
print()
print({k: round(v, 3) for k, v in model.score(table).items()})

Read the coefficients as directions, and remember these are **six scenes**,
not the hundred the paper fitted:

- **`log_size` positive** — bigger objects get more fixations. The largest
  and most reliable effect, here as in the paper.
- **`z_ecc` negative** — objects near the middle of the picture get more,
  matching the paper.
- **`log_depth`** — the paper finds nearer objects get more (a negative
  coefficient). On a handful of scenes this term is unstable and can even
  flip sign: whether the far half of a beach outdraws the near half of a
  bakery counter depends a lot on *which* six scenes you picked. Run the
  full hundred (section 7) before believing any depth story.

That is not a caveat to apologise for — it is the point of having the model
in a package. Refitting on subsets shows you immediately which effects are
robust and which are fragile.

The published fits are included, so you can see what you are up against:

In [ ]:
import pandas as pd
pd.DataFrame(oif.PUBLISHED_FIT).T

### Predictions, and where they fail

The interesting picture is not the prediction — it is the residual. Which
objects got far more attention than their size and position can explain?
Those are the ones where meaning is doing the work.

In [ ]:
table["predicted"] = model.predict(table, scale="count")
table["residual"] = table["n_fixations"] - table["predicted"]

one = table[table["image"] == scene.name]
fig, axes = plt.subplots(1, 3, figsize=(19, 5))
show_object_values(scene.label_map, dict(zip(one["mask_id"], one["n_fixations"])),
                   ax=axes[0], cmap="magma", title="observed", colorbar=False)
show_object_values(scene.label_map, dict(zip(one["mask_id"], one["predicted"])),
                   ax=axes[1], cmap="magma", title="predicted", colorbar=False)
show_object_values(scene.label_map, dict(zip(one["mask_id"], one["residual"].abs())),
                   ax=axes[2], cmap="inferno", title="absolute residual", colorbar=False)
plt.show()

one.reindex(one["residual"].abs().sort_values(ascending=False).index)[
    ["label", "size", "n_fixations", "predicted", "residual"]].head(5).round(1)

### Testing it honestly

Held-out **scenes**, not held-out objects. Objects within a scene share a
viewpoint, a depth range and the same viewers, so splitting at the object
level leaks information across the fold boundary and flatters the model.

In [ ]:
from oif.model import cross_validate_by_scene
cross_validate_by_scene(table, n_folds=3).round(3)

## 7. Using your own data

Three things to change.

**1. Get the whole dataset.** Uncomment and run — a few minutes and about
500 MB.

In [ ]:
# !git clone --depth 1 https://github.com/ehhall/objects-in-focus.git full-dataset
# data = OiF("full-dataset")
# !cd full-dataset && oif check && oif repair && oif labels

**2. Add your fixations.** The released `raw/*.npy` maps come with the full
dataset, so the memorization fixations are already there. To analyse your own
study instead, put your eye-tracker's CSV exports in `raw/` next to `images/`,
then `data.fixations()` reads all of them — it handles the two formats side by
side. SR Research
DataViewer exports (`CURRENT_FIX_X`, `CURRENT_FIX_Y`, …) and the older
`locs_1`/`locs_2`/`durs` layout are recognised automatically. For anything
else, say which column is which:

```python
from oif import load_fixations

fixations = load_fixations("raw/", columns={"x": "gaze_x", "y": "gaze_y",
                                            "image": "stimulus",
                                            "subject": "participant"})
```

Whatever the input looks like, the output is always the same table:
`subject, image, fix_index, x, y, duration, task`. The `image` column has to
match the scene names — `target_bakery`, with or without `.png`.

If your coordinates are in screen pixels rather than image pixels:

```python
from oif.mapping import rescale_fixations
fixations = rescale_fixations(fixations, from_shape=(1050, 1680),
                              to_shape=(768, 1024), mode="fit")
```

**3. Add salience, if you want that predictor.** Maps are not bundled —
generate them with [DeepGaze](https://github.com/matthias-k/DeepGaze)
(DeepGaze IIE produced the published numbers;
[DeepGaze III](https://doi.org/10.1167/jov.22.5.7) is the current model), save
one `.npy` per scene, and load them:

```python
from oif.salience import load_salience_maps
salience = load_salience_maps("salience/", shape=(768, 1024))
table = data.object_tables(fixations, salience=salience)
```

Everything runs without them; you just lose that one term.

## Doing it without writing code

The package installs a command line too:

```bash
oif check                          # is my copy of the data complete?
oif labels                         # write derived/labels.csv - what each mask id is
oif repair                         # rebuild the mask files that shipped truncated
oif objects --fixations raw/ --method disc --radius 25 --out objects.csv
oif demo target_bakery --labels    # save a picture of one scene's objects
```

`oif objects` writes exactly the table this notebook built, in one line.

## Where to go next

- **[The package tour](https://colab.research.google.com/github/ehhall/objects-in-focus/blob/main/notebooks/package_tour.ipynb)** — one section per module: what every function is for and how to call it
- **[Project website](https://ehhall.github.io/objects-in-focus/)** — figures from the paper and the poster
- **[Dataset reference](https://github.com/ehhall/objects-in-focus/blob/main/docs/dataset.md)** — every file and array convention
- **[How mask ids were matched to objects](https://github.com/ehhall/objects-in-focus/blob/main/docs/labels.md)**
- **[API reference](https://github.com/ehhall/objects-in-focus/blob/main/docs/api.md)**
- **[COCO-Freeview](https://github.com/ehhall/objects-in-focus/blob/main/docs/coco.md)** — the same analysis on MS-COCO

Something not working, or a polygon that looks wrong?
[Open an issue](https://github.com/ehhall/objects-in-focus/issues) — questions
about the data usually mean the documentation has a hole in it.

If you use the dataset or the code, please cite:

> Hall, E. H., & Loh, Z. (2025). Objects in Focus: Predicting Object-Based
> Attention from Spatial Features. *ICCV 2025 Workshop on Human-Inspired
> Computer Vision.*